# train_cloud_model

Этот ноутбук обучает cloud-модель на XGBoost с SMOTE.

Что делает ноутбук:
- загружает датасет через `build_cloud_dataset`
- делит данные на train / test
- применяет SMOTE только к train
- обучает `XGBClassifier`
- считает метрики
- сохраняет модель и артефакты
- сохраняет test set в CSV


In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

from imblearn.over_sampling import SMOTE

from preprocess import build_cloud_dataset, save_feature_list, CLOUD_FEATURES

## 1. Пути и конфиг

In [2]:
ARTIFACTS_DIR = Path("training/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "cloud_model.pkl"
FEATURES_PATH = ARTIFACTS_DIR / "cloud_feature_list.json"
METRICS_PATH = ARTIFACTS_DIR / "cloud_metrics.json"
TEST_CSV_PATH = ARTIFACTS_DIR / "cloud_test_set.csv"
TEST_PREDICTIONS_CSV_PATH = ARTIFACTS_DIR / "cloud_test_predictions.csv"

## 2. Загрузка датасета

In [3]:
X, y = build_cloud_dataset()

print("Dataset shape:", X.shape)
print("\nOriginal class distribution:")
print(y.value_counts())

X.head()

Dataset shape: (10000, 6)

Original class distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min]
0,298.1,10.5,1551,42.8,6.951591,0
1,298.2,10.5,1408,46.3,6.826723,3
2,298.1,10.4,1498,49.4,7.749388,5
3,298.2,10.4,1433,39.5,5.927505,7
4,298.2,10.5,1408,40.0,5.897817,9


## 3. Train / test split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution BEFORE SMOTE:")
print(y_train.value_counts())

Train shape: (8000, 6)
Test shape: (2000, 6)

Train distribution BEFORE SMOTE:
Machine failure
0    7729
1     271
Name: count, dtype: int64


## 4. Сохраняем test set в CSV

In [5]:
test_df = X_test.copy()
test_df["Machine failure"] = y_test.values
test_df.to_csv(TEST_CSV_PATH, index=False)

print("Saved test set to:", TEST_CSV_PATH)
test_df.head()


Saved test set to: training\artifacts\cloud_test_set.csv


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min],Machine failure
2997,300.5,9.3,1345,62.7,8.831174,153,0
4871,303.7,8.7,1513,40.1,6.353484,135,0
3858,302.5,8.9,1559,37.6,6.138504,209,0
951,295.6,10.7,1509,35.8,5.657192,60,0
6463,300.5,9.5,1358,60.4,8.589449,102,0


## 5. SMOTE только на train

In [6]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Train shape AFTER SMOTE:", X_train_resampled.shape)
print("\nTrain distribution AFTER SMOTE:")
print(pd.Series(y_train_resampled).value_counts())


Train shape AFTER SMOTE: (15458, 6)

Train distribution AFTER SMOTE:
Machine failure
0    7729
1    7729
Name: count, dtype: int64


In [7]:
SAFE_FEATURE_MAP = {
    "Air temperature [K]": "air_temperature_k",
    "temp_diff": "temp_diff",
    "Rotational speed [rpm]": "rotational_speed_rpm",
    "Torque [Nm]": "torque_nm",
    "power_kw": "power_kw",
    "Tool wear [min]": "tool_wear_min",
}

X_train_resampled = X_train_resampled.rename(columns=SAFE_FEATURE_MAP)
X_test = X_test.rename(columns=SAFE_FEATURE_MAP)

## 6. Обучение XGBoost

In [8]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=1.0,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_resampled, y_train_resampled)
print("Cloud model trained successfully")

Cloud model trained successfully


## 7. Предсказания и метрики

In [9]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "model_type": "cloud_binary",
    "model_name": "XGBoost + SMOTE",
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 6),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 6),
    "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 6),
    "f1_score": round(float(f1_score(y_test, y_pred, zero_division=0)), 6),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 6),
    "pr_auc": round(float(average_precision_score(y_test, y_prob)), 6),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "features": list(CLOUD_FEATURES),
    "train_size_before_smote": int(len(X_train)),
    "train_size_after_smote": int(len(X_train_resampled)),
    "test_size": int(len(X_test)),
}

print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nMetrics:")
print(json.dumps(metrics, indent=2))

Classification report:
              precision    recall  f1-score   support

           0     0.9952    0.9695    0.9822      1932
           1     0.5000    0.8676    0.6344        68

    accuracy                         0.9660      2000
   macro avg     0.7476    0.9186    0.8083      2000
weighted avg     0.9784    0.9660    0.9703      2000


Metrics:
{
  "model_type": "cloud_binary",
  "model_name": "XGBoost + SMOTE",
  "accuracy": 0.966,
  "precision": 0.5,
  "recall": 0.867647,
  "f1_score": 0.634409,
  "roc_auc": 0.974775,
  "pr_auc": 0.845945,
  "confusion_matrix": [
    [
      1873,
      59
    ],
    [
      9,
      59
    ]
  ],
  "features": [
    "Air temperature [K]",
    "temp_diff",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "power_kw",
    "Tool wear [min]"
  ],
  "train_size_before_smote": 8000,
  "train_size_after_smote": 15458,
  "test_size": 2000
}


## 8. Сохраняем test predictions в CSV

In [10]:
test_predictions_df = X_test.copy()
test_predictions_df["Machine failure"] = y_test.values
test_predictions_df["predicted_label"] = y_pred
test_predictions_df["predicted_probability"] = y_prob
test_predictions_df.to_csv(TEST_PREDICTIONS_CSV_PATH, index=False)

print("Saved test predictions to:", TEST_PREDICTIONS_CSV_PATH)
test_predictions_df.head()


Saved test predictions to: training\artifacts\cloud_test_predictions.csv


,air_temperature_k,temp_diff,rotational_speed_rpm,torque_nm,power_kw,tool_wear_min,Machine failure,predicted_label,predicted_probability
2997,300.5,9.3,1345,62.7,8.831174,153,0,0,0.282173
4871,303.7,8.7,1513,40.1,6.353484,135,0,0,0.001463
3858,302.5,8.9,1559,37.6,6.138504,209,0,1,0.534482
951,295.6,10.7,1509,35.8,5.657192,60,0,0,0.000720
6463,300.5,9.5,1358,60.4,8.589449,102,0,0,0.341708


## 9. Важность признаков

In [11]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance

,feature,importance
2,Rotational speed [rpm],0.244621
4,power_kw,0.239782
3,Torque [Nm],0.209243
5,Tool wear [min],0.189275
1,temp_diff,0.072340
0,Air temperature [K],0.044740


## 10. Сохранение модели и артефактов

In [12]:
joblib.dump(model, MODEL_PATH)
save_feature_list(list(CLOUD_FEATURES), FEATURES_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved feature list to:", FEATURES_PATH)
print("Saved metrics to:", METRICS_PATH)


Feature list saved to: training\artifacts\cloud_feature_list.json
Saved model to: training\artifacts\cloud_model.pkl
Saved feature list to: training\artifacts\cloud_feature_list.json
Saved metrics to: training\artifacts\cloud_metrics.json


## 11. Пример одного cloud prediction

In [13]:
sample_input = X_test.iloc[[0]].copy()
sample_risk = float(model.predict_proba(sample_input)[0, 1])
sample_pred = int(model.predict(sample_input)[0])

result = {
    "risk_score": round(sample_risk, 4),
    "prediction": "HIGH_RISK" if sample_risk >= 0.5 else "LOW_RISK",
    "features": sample_input.to_dict(orient="records")[0],
}

print(json.dumps(result, indent=2))


{
  "risk_score": 0.2822,
  "prediction": "LOW_RISK",
  "features": {
    "air_temperature_k": 300.5,
    "temp_diff": 9.300000000000011,
    "rotational_speed_rpm": 1345,
    "torque_nm": 62.7,
    "power_kw": 8.831174028873589,
    "tool_wear_min": 153
  }
}
